In [ ]:
USE WAREHOUSE VSENTISHCHEVA_WH;
USE DATABASE VSENTISHCHEVA_DB;
USE SCHEMA taxi;

In [ ]:
-- Create storage integration --
CREATE OR REPLACE STORAGE INTEGRATION VSENTISHCHEVA_S3_int
  TYPE = EXTERNAL_STAGE
  STORAGE_PROVIDER = 'S3'
  ENABLED = TRUE
  STORAGE_AWS_ROLE_ARN = 'arn:aws:iam::554739427960:role/snowflake-read-only-vsentishcheva'
  STORAGE_ALLOWED_LOCATIONS = ('*');  
  
GRANT USAGE ON INTEGRATION VSENTISHCHEVA_S3_int to SYSADMIN;

DESC INTEGRATION VSENTISHCHEVA_S3_int;

In [ ]:
-- create csv format

CREATE OR REPLACE FILE FORMAT csv_format
    TYPE = 'CSV'
    COMPRESSION = 'AUTO'
    FIELD_DELIMITER = ','
    RECORD_DELIMITER = '\n'
    SKIP_HEADER = 1
    TRIM_SPACE = TRUE,
    FIELD_OPTIONALLY_ENCLOSED_BY='"'
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE;

In [ ]:
-- create parquet format
CREATE OR REPLACE FILE FORMAT parquet_format
    TYPE = 'PARQUET'
    COMPRESSION = AUTO
    USE_LOGICAL_TYPE = TRUE
    TRIM_SPACE = TRUE;

In [ ]:
-- Create external stage
CREATE OR REPLACE STAGE vsentishcheva_s3_stage
  STORAGE_INTEGRATION = VSENTISHCHEVA_S3_int
  URL = 's3://vsentishcheva-hw/nyc_taxi/'
  FILE_FORMAT = csv_format;


LIST @vsentishcheva_s3_stage;

In [ ]:
-- Create tables

CREATE TABLE IF NOT EXISTS yellow_raw(
    VendorID BIGINT,
    passenger_count BIGINT,
    trip_distance DOUBLE,
    RatecodeID BIGINT,
    store_and_fwd_flag VARCHAR,
    PULocationID BIGINT,
    DOLocationID BIGINT,
    payment_type BIGINT,
    fare_amount DOUBLE,
    extra DOUBLE,
    mta_tax DOUBLE,
    tip_amount DOUBLE,
    tolls_amount DOUBLE,
    improvement_surcharge DOUBLE,
    total_amount DOUBLE,
    congestion_surcharge DOUBLE,
    tpep_pickup_datetime TIMESTAMP_NTZ,
    tpep_dropoff_datetime TIMESTAMP_NTZ,
    Airport_fee DOUBLE
);

CREATE TABLE IF NOT EXISTS green_raw(
    VendorID BIGINT,
    passenger_count BIGINT,
    trip_distance DOUBLE,
    RatecodeID BIGINT,
    store_and_fwd_flag VARCHAR,
    PULocationID BIGINT,
    DOLocationID BIGINT,
    payment_type BIGINT,
    fare_amount DOUBLE,
    extra DOUBLE,
    mta_tax DOUBLE,
    tip_amount DOUBLE,
    tolls_amount DOUBLE,
    improvement_surcharge DOUBLE,
    total_amount DOUBLE,
    congestion_surcharge DOUBLE,
    lpep_pickup_datetime TIMESTAMP_NTZ,
    lpep_dropoff_datetime TIMESTAMP_NTZ
);

CREATE TABLE IF NOT EXISTS taxi_zone_lookup(
    LocationID BIGINT,
    Borough VARCHAR,
    Zone VARCHAR,
    service_zone VARCHAR
);

In [ ]:
-- Load data
COPY INTO taxi.yellow_raw
    FROM @vsentishcheva_s3_stage/yellow
    FILE_FORMAT = parquet_format
    MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
    ON_ERROR = 'CONTINUE';
    
COPY INTO taxi.taxi_zone_lookup
    FROM @vsentishcheva_s3_stage/taxi_zone_lookup.csv
    FILE_FORMAT = csv_format
    ON_ERROR = 'CONTINUE';

COPY INTO taxi.green_raw
    FROM @vsentishcheva_s3_stage/green
    FILE_FORMAT = parquet_format
    MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
    ON_ERROR = 'CONTINUE';

In [ ]:
-- Enrich tables
CREATE OR REPLACE TABLE yellow_enriched AS
SELECT
    y.*,
    pu.Zone AS pickup_zone,
    do.Zone AS dropoff_zone
FROM yellow_raw y
LEFT JOIN taxi_zone_lookup pu ON y.PULocationID = pu.LocationID
LEFT JOIN taxi_zone_lookup do ON y.DOLocationID = do.LocationID;

CREATE OR REPLACE TABLE green_enriched AS
SELECT
    g.*,
    pu.Zone AS pickup_zone,
    do.Zone AS dropoff_zone
FROM green_raw g
LEFT JOIN taxi_zone_lookup pu ON g.PULocationID = pu.LocationID
LEFT JOIN taxi_zone_lookup do ON g.DOLocationID = do.LocationID;

In [ ]:
-- Create aggregated tables
CREATE OR REPLACE TABLE yellow_aggregated AS
WITH yellow_transformed AS (
    SELECT
        *,
        CASE 
            WHEN trip_distance <= 2 THEN 'Short'
            WHEN trip_distance <= 10 THEN 'Medium'
            ELSE 'Long'
        END AS trip_category,
        DATE_PART('HOUR', tpep_pickup_datetime) AS pickup_hour
    FROM yellow_enriched
    WHERE trip_distance > 0
      AND total_amount > 0
      AND passenger_count BETWEEN 1 AND 6
)
SELECT
    pickup_zone,
    ROUND(AVG(trip_distance), 3)   AS avg_trip_distance,
    ROUND(AVG(total_amount), 2)    AS avg_total_amount,
    ROUND(AVG(tip_amount), 2)      AS avg_tip_amount,
    ROUND(MAX(trip_distance), 3)   AS max_trip_distance,
    ROUND(MIN(tip_amount), 2)      AS min_tip_amount,
    COUNT(*)                       AS total_trips
FROM yellow_transformed
GROUP BY pickup_zone;

CREATE OR REPLACE TABLE green_aggregated AS
WITH green_transformed AS (
    SELECT
        *,
        CASE 
            WHEN trip_distance <= 2 THEN 'Short'
            WHEN trip_distance <= 10 THEN 'Medium'
            ELSE 'Long'
        END AS trip_category,
        DATE_PART('HOUR', lpep_pickup_datetime) AS pickup_hour
    FROM green_enriched
    WHERE trip_distance > 0
      AND total_amount > 0
      AND passenger_count BETWEEN 1 AND 6
)
SELECT
    pickup_zone,
    ROUND(AVG(trip_distance), 3)   AS avg_trip_distance,
    ROUND(AVG(total_amount), 2)    AS avg_total_amount,
    ROUND(AVG(tip_amount), 2)      AS avg_tip_amount,
    ROUND(MAX(trip_distance), 3)   AS max_trip_distance,
    ROUND(MIN(tip_amount), 2)      AS min_tip_amount,
    COUNT(*)                       AS total_trips
FROM green_transformed
GROUP BY pickup_zone;

Time Travel

In [ ]:
-- Delete data
DELETE FROM green_enriched
WHERE trip_distance = 0 AND passenger_count = 9 and PULocationID = 181;

In [ ]:
-- Check previous version of table
SELECT * 
FROM green_enriched AT (OFFSET => -60*5)
WHERE trip_distance = 0 AND passenger_count = 9 and PULocationID = 181;

In [ ]:
-- Restore to a new table
CREATE OR REPLACE TABLE green_recovered AS 
SELECT * 
FROM green_enriched  AT (OFFSET => -60*20);

In [ ]:
-- Check results
SELECT * 
FROM green_recovered
WHERE trip_distance = 0 AND passenger_count = 9 and PULocationID = 181;

Streams

In [ ]:
-- Create strim
CREATE OR REPLACE STREAM yellow_stream ON TABLE yellow_enriched;

In [ ]:
SELECT *
FROM yellow_stream;

In [ ]:
-- Add new data
INSERT INTO yellow_enriched VALUES (1, 12, 0, 5,'N', 107, 1, 1, 100, 0, 0, 0, 0, 0, 100, 0, CURRENT_TIMESTAMP, CURRENT_TIMESTAMP, 0, 'Midtown', 'JFK Airport');

In [ ]:
-- Check changes
SELECT * FROM yellow_stream;

In [ ]:
-- Create table for log
CREATE OR REPLACE TABLE yellow_changes_log LIKE yellow_enriched;

TASKS

In [ ]:
-- Task for changes log
CREATE OR REPLACE TASK yellow_stream_task
  WAREHOUSE = VSENTISHCHEVA_WH
  SCHEDULE = '1 HOUR'
AS
INSERT INTO yellow_changes_log
SELECT * EXCLUDE(METADATA$ACTION, METADATA$ISUPDATE, METADATA$ROW_ID)
FROM yellow_stream;

In [ ]:
ALTER TASK yellow_stream_task RESUME;

In [ ]:
-- Add new data
INSERT INTO yellow_enriched VALUES (1, 10, 0, 5,'N', 107, 1, 1, 100, 0, 0, 0, 0, 0, 100, 0, CURRENT_TIMESTAMP, CURRENT_TIMESTAMP, 0, 'Midtown', 'JFK Airport');

In [ ]:
EXECUTE TASK yellow_stream_task;

In [ ]:
SELECT *
FROM yellow_changes_log;

In [ ]:
-- table for aggregated hourly stats
CREATE OR REPLACE TABLE zone_hourly_stats (
    pickup_zone VARCHAR,
    datetime TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    avg_distance FLOAT,
    avg_total_amount FLOAT,
    total_trips INT
);

-- task for aggregated stats
CREATE OR REPLACE TASK zone_stats_task
  WAREHOUSE = VSENTISHCHEVA_WH
  SCHEDULE = '1 HOUR'
AS
INSERT INTO zone_hourly_stats
SELECT
    pickup_zone,
    CURRENT_TIMESTAMP as datetime,
    ROUND(AVG(trip_distance),3) AS avg_distance,
    ROUND(AVG(total_amount),2) AS avg_total_amount,
    COUNT (*) AS total_trips
FROM yellow_enriched
GROUP BY pickup_zone;


In [ ]:
ALTER TASK zone_stats_task RESUME;

In [ ]:
EXECUTE TASK zone_stats_task;

In [ ]:
SELECT *
FROM zone_hourly_stats
LIMIT 10;

Additional task

In [ ]:
-- Union yellow and green tables
CREATE OR REPLACE TABLE all_trips AS
SELECT
    VendorID,
    passenger_count,
    trip_distance,
    RatecodeID,
    store_and_fwd_flag,
    PULocationID,
    DOLocationID,
    payment_type,
    fare_amount,
    extra,
    mta_tax,
    tip_amount,
    tolls_amount,
    improvement_surcharge,
    total_amount,
    congestion_surcharge,
    tpep_pickup_datetime AS pickup_datetime,
    tpep_dropoff_datetime AS dropoff_datetime,
    Airport_fee,
    pickup_zone,
    dropoff_zone,
    'yellow' AS taxi_type
FROM yellow_enriched
UNION ALL
SELECT
    VendorID,
    passenger_count,
    trip_distance,
    RatecodeID,
    store_and_fwd_flag,
    PULocationID,
    DOLocationID,
    payment_type,
    fare_amount,
    extra,
    mta_tax,
    tip_amount,
    tolls_amount,
    improvement_surcharge,
    total_amount,
    congestion_surcharge,
    lpep_pickup_datetime AS pickup_datetime,
    lpep_dropoff_datetime AS dropoff_datetime,
    NULL AS Airport_fee,
    pickup_zone,
    dropoff_zone,
    'green' AS taxi_type
FROM green_enriched;

In [ ]:
-- Table for logs
CREATE OR REPLACE TABLE load_logs (
    log_id INT AUTOINCREMENT,
    inserted_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    status VARCHAR
);

In [ ]:
-- stored procedure
CREATE OR REPLACE PROCEDURE insert_trip(
      VENDORID              NUMBER,
      PASSENGER_COUNT       NUMBER,
      TRIP_DISTANCE         FLOAT,
      RATECODEID            NUMBER,
      STORE_AND_FWD_FLAG    VARCHAR,
      PULOCATIONID          NUMBER,
      DOLOCATIONID          NUMBER,
      PAYMENT_TYPE          NUMBER,
      FARE_AMOUNT           FLOAT,
      EXTRA                 FLOAT,
      MTA_TAX               FLOAT,
      TIP_AMOUNT            FLOAT,
      TOLLS_AMOUNT          FLOAT,
      IMPROVEMENT_SURCHARGE FLOAT,
      TOTAL_AMOUNT          FLOAT,
      CONGESTION_SURCHARGE  FLOAT,
      PICKUP_DATETIME       TIMESTAMP_NTZ,
      DROPOFF_DATETIME      TIMESTAMP_NTZ,
      AIRPORT_FEE           FLOAT,
      PICKUP_ZONE           VARCHAR,
      DROPOFF_ZONE          VARCHAR,
      TAXI_TYPE             VARCHAR
)
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
DECLARE
    CNT INT;
    result VARCHAR;
BEGIN
    SELECT COUNT(1) INTO :CNT
    FROM ALL_TRIPS T
    WHERE T.VENDORID              = :VENDORID
      AND T.PASSENGER_COUNT       = :PASSENGER_COUNT
      AND T.TRIP_DISTANCE         = :TRIP_DISTANCE
      AND T.RATECODEID            = :RATECODEID
      AND T.STORE_AND_FWD_FLAG    = :STORE_AND_FWD_FLAG
      AND T.PULOCATIONID          = :PULOCATIONID
      AND T.DOLOCATIONID          = :DOLOCATIONID
      AND T.PAYMENT_TYPE          = :PAYMENT_TYPE
      AND T.FARE_AMOUNT           = :FARE_AMOUNT
      AND T.EXTRA                 = :EXTRA
      AND T.MTA_TAX               = :MTA_TAX
      AND T.TIP_AMOUNT            = :TIP_AMOUNT
      AND T.TOLLS_AMOUNT          = :TOLLS_AMOUNT
      AND T.IMPROVEMENT_SURCHARGE = :IMPROVEMENT_SURCHARGE
      AND T.TOTAL_AMOUNT          = :TOTAL_AMOUNT
      AND T.CONGESTION_SURCHARGE  = :CONGESTION_SURCHARGE
      AND T.PICKUP_DATETIME       = :PICKUP_DATETIME
      AND T.DROPOFF_DATETIME      = :DROPOFF_DATETIME
      AND T.AIRPORT_FEE           = :AIRPORT_FEE
      AND T.PICKUP_ZONE           = :PICKUP_ZONE
      AND T.DROPOFF_ZONE          = :DROPOFF_ZONE
      AND T.TAXI_TYPE             = :TAXI_TYPE;

    IF (CNT = 0) THEN
        INSERT INTO ALL_TRIPS (
            VENDORID, PASSENGER_COUNT, TRIP_DISTANCE, RATECODEID,
            STORE_AND_FWD_FLAG, PULOCATIONID, DOLOCATIONID, PAYMENT_TYPE,
            FARE_AMOUNT, EXTRA, MTA_TAX, TIP_AMOUNT, TOLLS_AMOUNT,
            IMPROVEMENT_SURCHARGE, TOTAL_AMOUNT, CONGESTION_SURCHARGE,
            PICKUP_DATETIME, DROPOFF_DATETIME, AIRPORT_FEE,
            PICKUP_ZONE, DROPOFF_ZONE, TAXI_TYPE
        )
        VALUES (
            :VENDORID, :PASSENGER_COUNT, :TRIP_DISTANCE, :RATECODEID,
            :STORE_AND_FWD_FLAG, :PULOCATIONID, :DOLOCATIONID, :PAYMENT_TYPE,
            :FARE_AMOUNT, :EXTRA, :MTA_TAX, :TIP_AMOUNT, :TOLLS_AMOUNT,
            :IMPROVEMENT_SURCHARGE, :TOTAL_AMOUNT, :CONGESTION_SURCHARGE,
            :PICKUP_DATETIME, :DROPOFF_DATETIME, :AIRPORT_FEE,
            :PICKUP_ZONE, :DROPOFF_ZONE, :TAXI_TYPE
        );
        result := 'INSERTED';
    ELSE
        result := 'DUPLICATE';
    END IF;

    
    INSERT INTO load_logs(STATUS)
    VALUES (:result);

    RETURN result;
END;
$$;


In [ ]:
CALL insert_trip(1, 10, 0, 5,'N', 107, 1, 1, 100, 0, 0, 0, 0, 0, 100, 0, '2025-08-18 09:00:00', '2025-08-18 09:00:00', 0, 'Midtown', 'JFK Airport', 'yellow');

In [ ]:
Select *
FROM load_logs;

In [ ]:
-- Zero-copy cloning
CREATE OR REPLACE DATABASE VSENTISHCHEVA_taxi_dev CLONE VSENTISHCHEVA_DB;